# Reference · How scikit-learn is put together

Meeting 2 built a pipeline and ran short. This is the part that got squeezed: not
*what to type*, but **what the objects are**, so that the rest of the semester reads as
variations on one design rather than a pile of new syntax.

You already know model selection, fitting and scoring as statistical ideas. Nothing
here is new statistics. It is the shape of the library — and scikit-learn has an
unusually small design, which is why it is worth twenty minutes once instead of
guessing for fifteen weeks.

Run it, poke at it, come back to it when something will not fit together.

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:
    sys.path.insert(0, str(found[0] / "course"))
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")

import numpy as np
import pandas as pd
from stat764 import load

ames = load("ames.csv")
print(f"{len(ames)} houses, {ames.shape[1]} columns")

## 1. There is only one kind of object

Every model, every preprocessing step, every search procedure in scikit-learn is an
**estimator**, and they all speak the same four verbs. You will never meet a fifth.

| verb | what it does | who has it |
|---|---|---|
| `.fit(X, y)` | learn from data | everything |
| `.predict(X)` | produce predictions | models |
| `.transform(X)` | produce a modified `X` | preprocessors |
| `.fit_transform(X)` | both, in one call | preprocessors |

Classifiers add `.predict_proba(X)`. That is the entire vocabulary.

The payoff is **substitutability**: anything with `.fit`/`.predict` can go anywhere a
model goes. That is why Meeting 3 can swap OLS, ridge and a tree through the same
three lines, and why the bake-off in Week 9 is a `for` loop rather than a rewrite.

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor

X = ames[["Gr_Liv_Area", "Year_Built", "Overall_Qual"]]
y = ames["SalePrice"]

for model in [LinearRegression(), Ridge(alpha=10), DecisionTreeRegressor(max_depth=4)]:
    fitted = model.fit(X, y)                     # identical call, three different models
    print(f"  {type(model).__name__:<24} first prediction {fitted.predict(X)[:1][0]:>10,.0f}")

⚠ Those numbers are fitted on the same data they are scored on, which Meeting 1 spent
an hour warning you about. They are here to show the *interface* is identical, nothing
more.

## 2. Two families, told apart by what you can call

- A **predictor** has `.predict()`. `LinearRegression`, `Ridge`, `DecisionTreeRegressor`.
- A **transformer** has `.transform()`. `StandardScaler`, `SimpleImputer`, `OneHotEncoder`.

That is the only distinction that matters structurally. A transformer takes an `X` and
gives back a different `X`; a predictor takes an `X` and gives back a `y`.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
Z = scaler.fit_transform(X)
print(f"  in  {X.shape}   out {Z.shape}      <- same shape, different numbers")
print(f"  column means before: {X.mean().round(1).tolist()}")
print(f"  column means after : {Z.mean(axis=0).round(6).tolist()}")

## 3. The trailing underscore is the most useful convention in the library

Two kinds of number live on a fitted estimator, and telling them apart is most of
understanding what a model *is*:

| | set by | example |
|---|---|---|
| **Hyperparameter** | you, in the constructor | `Ridge(alpha=10)` → `.alpha` |
| **Learned parameter** | `.fit()`, from the data | `.coef_`, `.mean_`, `.categories_` |

**Anything ending in `_` was estimated from data.** Nothing ending in `_` exists before
you call `.fit()`.

That is not a naming quirk — it is the line between "a choice you have to justify" and
"an estimate with uncertainty attached." Week 4 is entirely about the first column:
ridge and lasso on the Tuesday, and on the Thursday, choosing the penalty without
letting the second column cheat.

In [ ]:
ridge = Ridge(alpha=10)
print("  before fit — hyperparameters exist, learned state does not")
print(f"    alpha = {ridge.alpha}      has coef_? {hasattr(ridge, 'coef_')}")

ridge.fit(X, y)
print("  after fit")
print(f"    alpha = {ridge.alpha}      has coef_? {hasattr(ridge, 'coef_')}")
print(f"    coef_ = {ridge.coef_.round(1).tolist()}")

print("\n  every hyperparameter, in one call:")
print("   ", ridge.get_params())

## 4. `Pipeline` — a chain that behaves like one estimator

A pipeline is a list of `(name, estimator)` steps. Every step but the last is a
transformer; the last is your model. The whole thing then has `.fit()` and `.predict()`
like any other estimator — which is the entire trick.

**The asymmetry is the point:**

```
pipe.fit(X_train, y_train)   ->  fit_transform each step, then fit the model
pipe.predict(X_test)         ->  transform each step (NO refit), then predict
```

A scaler that has been fit knows the training means and SDs. When test data goes
through, it is scaled *by the training constants* — not by its own. That is what makes
the held-out score honest, and it is the thing that is easy to get wrong by hand and
impossible to get wrong here.

In [ ]:
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=764)

pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", Ridge(alpha=10)),
])
pipe.fit(X_train, y_train)
print(f"  R-squared on unseen data: {r2_score(y_test, pipe.predict(X_test)):.3f}")

print("\n  the scaler learned these from the TRAINING rows only:")
print(f"    means {pipe.named_steps['scale'].mean_.round(1).tolist()}")
print(f"    (train means {X_train.mean().round(1).tolist()})")
print(f"    (test  means {X_test.mean().round(1).tolist()}  <- never used)")

## 5. `ColumnTransformer` — different treatment for different columns

You cannot median-impute a neighbourhood, and you cannot standardise a string. A
`ColumnTransformer` sends named column lists to different transformers and glues the
results back together **side by side**.

Each branch is `(name, transformer, columns)` — and `transformer` can itself be a
`Pipeline`, which is why these grow nested.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numeric     = ["Gr_Liv_Area", "Year_Built", "Overall_Qual", "Total_Bsmt_SF"]
categorical = ["Neighborhood", "Central_Air", "Garage_Type"]

prep = ColumnTransformer([
    ("num", Pipeline([("fill",  SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]),                       numeric),
    ("cat", Pipeline([("fill",  SimpleImputer(strategy="constant",
                                              fill_value="Missing")),
                      ("encode", OneHotEncoder(handle_unknown="ignore",
                                               sparse_output=False))]),    categorical),
])

A_train, A_test, b_train, b_test = train_test_split(
    ames[numeric + categorical], y, test_size=0.25, random_state=764)
out = prep.fit_transform(A_train)
print(f"  {A_train.shape[1]} columns in  ->  {out.shape[1]} columns out")
print(f"    {len(numeric)} numeric, scaled")
for name, cats in zip(categorical, prep.named_transformers_['cat']
                                       .named_steps['encode'].categories_):
    print(f"    {name:<14} -> {len(cats)} dummy columns")

⚠ Two things that catch people:

1. **Columns you do not name are dropped.** `remainder="drop"` is the default. Pass
   `remainder="passthrough"` if you meant to keep them.
2. `Garage_Type` produces **7** dummies from 6 real levels. The imputer filled its 119
   blanks with the string `"Missing"`, and the encoder treated that as a category of its
   own. That is deliberate — a missing garage type almost certainly means *no garage*,
   which is a fact about the house rather than an absent measurement.

## 6. Coming from R: where did the reference level go?

In R, `lm(y ~ f)` with a k-level factor gives you **k − 1** dummies. One level is the
**reference**, absorbed into the intercept, and every coefficient reads as "this level
minus the reference." That is `contr.treatment`, R's default, and it keeps the design
matrix full rank.

`OneHotEncoder` does **not** do that by default. It gives you **all k columns**, no
reference level. The rows then sum to 1, which is perfectly collinear with an intercept
— the dummy-variable trap, walked into deliberately.

This surprises people, and the surprise is worth resolving properly rather than
shrugging at, because it changes what your coefficients mean.

In [ ]:
qual = pd.cut(ames["Overall_Qual"], [0, 5, 7, 10], labels=["low", "mid", "high"])
Q = pd.DataFrame({"Qual": qual.astype(str)})
price = ames["SalePrice"]

full_enc = OneHotEncoder(sparse_output=False).fit(Q)
Zf = full_enc.transform(Q)
print("DEFAULT — one column per level")
print("  columns:", full_enc.get_feature_names_out().tolist())
print(f"  every row sums to 1? {np.allclose(Zf.sum(axis=1), 1)}   <- collinear with an intercept")

drop_enc = OneHotEncoder(drop="first", sparse_output=False).fit(Q)
Zd = drop_enc.transform(Q)
print("\ndrop='first' — R's treatment contrast")
print(f"  reference level: {full_enc.categories_[0][drop_enc.drop_idx_[0]]}")
print("  columns:", drop_enc.get_feature_names_out().tolist())

### It does not change a single prediction

In [ ]:
pf = LinearRegression().fit(Zf, price).predict(Zf)
pd_ = LinearRegression().fit(Zd, price).predict(Zd)
print(f"  largest difference in fitted values: {np.abs(pf - pd_).max():.6f}")
print("\n  `LinearRegression` solves by least squares through an SVD, which returns the")
print("  MINIMUM-NORM solution rather than raising on a rank-deficient matrix.")
print("  R would refuse to fit this. scikit-learn quietly picks one of the infinitely")
print("  many coefficient vectors that give identical predictions.")

### It does change what the coefficients mean

In [ ]:
mf = LinearRegression().fit(Zf, price)
md = LinearRegression().fit(Zd, price)
means = price.groupby(Q["Qual"]).mean()

def show(label, model, names):
    body = "  ".join(f"{n} {c:+,.1f}" for n, c in zip(names, model.coef_))
    print(f"  {label:<13} intercept {model.intercept_:>10,.1f}   {body}")


print("  group means:  " + "  ".join(f"{k} {v:,.1f}" for k, v in means.items()))
print()
show("drop='first'", md, drop_enc.get_feature_names_out())
show("all levels", mf, full_enc.get_feature_names_out())

print("\n  drop='first' is exactly R: the intercept IS the reference group's mean,")
print("    and each coefficient is that level minus the reference.")
print(f"  all levels is NOT arbitrary either — those coefficients sum to "
      f"{abs(mf.coef_.sum()):.1f}, and")
print(f"    the intercept is the unweighted mean of the group means "
      f"({means.mean():,.1f}). Minimum-norm")
print("    forces the solution orthogonal to the collinear direction, which imposes")
print("    sum-to-zero — so the default lands you on R's contr.sum, not contr.treatment.")

⚠ Take that seriously before Week 10. The coefficients from the default encoding are
not merely awkward to interpret — **they are not unique.** The solver chose one of an
infinite family by a criterion you never specified. "The coefficient on `mid` is
−22,000" is a statement about `lstsq`'s tie-breaking rule as much as about houses.

### And with a penalty, it changes the fit

This is the part that is not cosmetic. Ridge penalises the coefficients, so a different
parameterisation is a different penalty — and you get different predictions.

In [ ]:
for alpha in (1, 100):
    a = Ridge(alpha=alpha).fit(Zf, price).predict(Zf)
    b = Ridge(alpha=alpha).fit(Zd, price).predict(Zd)
    print(f"  alpha={alpha:<5} largest difference in fitted values: {np.abs(a - b).max():>10,.1f}")

print("\n  Under a penalty the ALL-LEVELS encoding is arguably the better one: every")
print("  level is treated symmetrically. Dropping one makes the reference level's effect")
print("  unpenalised and folded into the intercept, which is a modelling choice nobody")
print("  meant to make. Week 4 is where this starts to matter.")

### ⚠ One trap if you do switch to `drop="first"`

In [ ]:
import warnings

tr = pd.DataFrame({"g": ["a", "b", "c"]})
te = pd.DataFrame({"g": ["a", "unseen"]})
for kw, label in [({}, "default"), ({"drop": "first"}, "drop='first'")]:
    enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False, **kw).fit(tr)
    # sklearn warns here, and the warning is the point -- caught and printed so it
    # reads as the demonstration it is rather than as something going wrong.
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        Z = enc.transform(te)
    if caught and label == "default":
        print(f"  sklearn says: {str(caught[0].message).splitlines()[0]}\n")
    note = ("<-- IDENTICAL to reference level 'a'" if kw
            else "<-- distinct from every known level")
    print(f"  {label:<13} 'a' -> {Z[0].astype(int).tolist()}    "
          f"'unseen' -> {Z[1].astype(int).tolist()}  {note}")

With `drop="first"`, the reference level is encoded as all zeros — and so is an unknown
category. They become **indistinguishable**, so every unseen level is silently scored as
whichever level happened to be alphabetically first. On Ames, `Landmrk` has one sale and
will sit outside the training fold most of the time.

### Which to use

| You are doing | Use |
|---|---|
| A linear model and you want to read the coefficients the way R prints them | `drop="first"` |
| Ridge, lasso, anything penalised | the default — symmetric across levels |
| Trees, forests, boosting | it does not matter; no intercept, no collinearity |
| A two-level column and the second dummy annoys you | `drop="if_binary"` |

The meeting notebooks use the default with `handle_unknown="ignore"`, which is right for
what they do: the models are OLS, ridge and trees, and nobody reads the coefficients.

## 7. Reading a nested pipeline, and reaching inside

Read it as a tree, outside in:

```
Pipeline
├── "prep"  ColumnTransformer
│   ├── "num" → Pipeline(fill → scale)      on the numeric columns
│   └── "cat" → Pipeline(fill → encode)     on the categorical columns
└── "model" Ridge()
```

The names are yours to choose. They are not decoration: they are how you reach a step
afterwards, and how you address one in a hyperparameter search.

In [ ]:
full = Pipeline([("prep", prep), ("model", Ridge(alpha=10))])
full.fit(A_train, b_train)

print(f"  R-squared {r2_score(b_test, full.predict(A_test)):.3f}")
print(f"  steps: {list(full.named_steps)}")
print(f"  the model itself: {full.named_steps['model']}")
print(f"  also reachable by position: {full[-1]}")

names = full.named_steps["prep"].get_feature_names_out()
print(f"\n  what the model actually sees — {len(names)} features, first six:")
print("   ", list(names[:6]))

## 8. Addressing a step by name: the double underscore

`step__parameter` reaches into a pipeline. Nest it as deep as you like:
`prep__num__fill__strategy`.

You will not need this until Week 4, when tuning moves *inside* the resampling. It is
here so the syntax is not also new when the idea is.

In [ ]:
print("  change the model's penalty without rebuilding anything:")
full.set_params(model__alpha=1000).fit(A_train, b_train)
print(f"    alpha=1000 -> R-squared {r2_score(b_test, full.predict(A_test)):.3f}")
full.set_params(model__alpha=10).fit(A_train, b_train)
print(f"    alpha=10   -> R-squared {r2_score(b_test, full.predict(A_test)):.3f}")

print("\n  and the imputer three levels down:")
print(f"    {full.get_params()['prep__num__fill__strategy']}")

## 9. Why any of this matters: resampling for free

Here is the cash value. `cross_val_score` refits **whatever you hand it** on each
training fold. Hand it a bare model and only the model is refit — your scaler and your
imputer were fitted once, on everything, and every fold is contaminated.

Hand it a pipeline and *every step* is refit, ten times, correctly, with no effort and
nothing to remember.

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

cv = KFold(n_splits=10, shuffle=True, random_state=764)
folds = cross_val_score(full, ames[numeric + categorical], y, cv=cv, scoring="r2")
print(f"  10-fold CV  mean {folds.mean():.3f}   SD {folds.std():.3f}")
print("  every fold refit the imputer, the scaler, the encoder and the model.")

That is the sentence to keep:

⚠ **A pipeline is not tidiness. It is the object that makes your resampling honest.**

Meeting 3 uses exactly this, and Meeting 4 is an hour of what goes wrong without it.

## Five things to remember

| | |
|---|---|
| **1** | Everything is an estimator, and there are only four verbs. |
| **2** | Trailing underscore means *learned from data*. No underscore means *you chose it*. |
| **3** | `.fit()` learns; `.transform()` on new data applies what was learned, and does not relearn. |
| **4** | `ColumnTransformer` routes columns; `Pipeline` chains steps; they nest freely. |
| **5** | If any preprocessing happens outside the pipeline, your cross-validation is wrong. |

---

## Also in this neighborhood

📗 **The user guide on composite estimators** — scikit-learn's own §6.1. Short, and the
diagrams are good. It also covers `FeatureUnion` and `make_pipeline`, which generate the
step names for you when you cannot be bothered to.

📗 **`set_config(display="diagram")`.** Run it once and a pipeline renders as a clickable
diagram in Jupyter instead of a wall of text. Genuinely helpful when they get deep.

🚫 **Writing your own estimator.** Perfectly doable — inherit `BaseEstimator`, implement
`fit` and `predict` — and not needed in this course. Worth knowing it is available if
your capstone wants a model scikit-learn does not have.